In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================================
# CHANGE ONLY THIS
# ==========================================================

# MODEL_PATH = "./V4B_Final_Merged_Model"
# MODEL_PATH = "./V5B_Final_Merged_Model/"
# MODEL_PATH = "Qwen/Qwen3-8B"
MODEL_PATH = "./V5D_Final_Merged_Model/"

# ==========================================================


g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

SYSTEM_PROMPT = """
You are Compatifi V4B.

Extract ONLY stable long-term memories about user PEOPLE.

Keep:
- Long-term preferences
- Long-term goals
- Stable personality traits
- Persistent habits
- Profession
- Skills
- Communication preferences
- Ongoing projects
- Persistent health conditions

Ignore:
- Temporary emotions
- Greetings
- One-time events
- Short-term plans
- Casual conversation

Return ONLY valid JSON.
"""










print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("\nModel Loaded Successfully!\n")


def ask_model(conversation, relationship="Friend"):
    user_prompt = f"""Domain: relationship
Relationship: {relationship}
Instruction: Extract long-term other people memories

Conversation:
{conversation}
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
        )

    response = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    print(response)

Loading tokenizer...
Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [01:48<00:00, 21.60s/it]
Some parameters are on the meta device because they were offloaded to the disk and cpu.



Model Loaded Successfully!



In [3]:
conversation = """
Friend: I still cycle every morning before work.
User: That's impressive.

Friend: I'm also studying Japanese because I want to move to Tokyo someday.
User: You'll definitely get there.

Friend: I work as a software engineer at Microsoft.
"""

ask_model(conversation)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


FileNotFoundError: No such file or directory: model-00003-of-00005.safetensors

In [ ]:
# The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
# <think>

# </think>

# {"memories": ["Cycles every morning before work", "Studying Japanese with the goal of moving to Tokyo", "Works as a software engineer at Microsoft"]}]}